# ATML PA3 — Task 1 (Qwen3-0.6B)

Choose **Runtime → Change runtime type → A100 GPU**. This notebook runs the manual's five methods and seven evaluation conditions. You will need persistent storage with **80–100 GB free** to keep all FP32 full-model recovery states. The complete experiment is long; run methods/evaluations separately across sessions as needed.

The setup uses an isolated Python environment. It leaves Colab's notebook packages untouched and runs commands in subprocesses using that environment. Do not replace these commands with `%pip` in the kernel. Python 3.11–3.13 is supported. CPU implementation tests passed locally; actual A100 training and full benchmark results still need to be run.


## 1. Obtain the code

Upload `task1-colab.zip` (provided with the implementation) to Colab's files panel, or clone the repository after this implementation has been committed and pushed. The cell reuses an existing checkout; it never deletes or resets files.


In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile

PROJECT = Path('/content/28100053-ATML-PA3')
ARCHIVE = Path('/content/task1-colab.zip')
if not PROJECT.exists():
    if ARCHIVE.exists():
        with zipfile.ZipFile(ARCHIVE) as archive:
            # Accept only the expected archive root and reject path traversal.
            for entry in archive.namelist():
                target = (Path('/content') / entry).resolve()
                if not target.is_relative_to(PROJECT.resolve()):
                    raise ValueError(f'Unexpected archive entry: {entry}')
            archive.extractall('/content')
    else:
        subprocess.run(['git', 'clone', 'https://github.com/Bulibuh/28100053-ATML-PA3.git', str(PROJECT)], check=True)
assert (PROJECT / 'scripts/setup_colab.py').exists(), 'Upload the implementation ZIP, or publish the new code before cloning main.'
os.chdir(PROJECT)
print('Repository:', PROJECT)


## 2. Install and verify dependencies

Select a GPU runtime before running this cell. Installation checks the official CUDA PyTorch build, package consistency, a tiny Qwen3 FP16 AMP update, and the offline regression tests. If it fails, inspect `setup_colab.log`; do not continue to experiments. Re-run installation after a Colab runtime reset.


In [ ]:
import runpy
runpy.run_path('scripts/colab_runner.py')['run_logged']([sys.executable, 'scripts/setup_colab.py'], PROJECT)
PYTHON = PROJECT / '.venv-colab/bin/python'

import runpy
runner = runpy.run_path(str(PROJECT / 'scripts/colab_runner.py'))
pa3 = runner['make_runner'](PROJECT, PYTHON)


## 3. Mount persistent storage

Keep the artifact directory the same on every session. It contains recovery states, the fixed data/probe selections, SVD bases, logs, and measurements. A free 15 GB Drive account will not hold all required checkpoints. Model/dataset downloads are cached on local disk and may need redownloading after a runtime reset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.environ['PA3_ARTIFACT_ROOT'] = '/content/drive/MyDrive/ATML_PA3_task1'
os.environ['HF_HOME'] = '/content/hf-cache'
ARTIFACTS = Path(os.environ['PA3_ARTIFACT_ROOT'])
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Persistent artifacts:', ARTIFACTS)


## 4. Freeze data selection, revisions, and diagnostic probes

This selects the first 40,000 eligible examples in source order, excludes GSM8K test overlaps and overlength examples, and samples the fixed 64 diagnostic prompts before any results are inspected. Re-running verifies existing hashes rather than selecting new examples.


In [ ]:
pa3('task1', 'prepare')


## 5. Shared baseline (run once)

Complete zero-shot benchmark splits; GSM8K greedy non-thinking generation, stock stopping/filtering, 1,024 new tokens. The other three tasks use candidate likelihood with stock completion prompts and no chat template. `--batch-size 1` can reduce evaluation memory without altering training.


In [ ]:
pa3('task1', 'evaluate', '--condition', 'baseline', '--batch-size', '2')
pa3('task1', 'qualitative', '--condition', 'baseline')


## 6. Full fine-tuning

Each method uses 40,000 examples, 1,250 successful updates, microbatch 2, accumulation 16, seed 6304, FP16 AMP, and FP32 master/optimizer parameters. Recovery checkpoints are saved at 312/625/937/1250 updates. Evaluate the saved midpoint and final states after training; midpoint evaluation uses update 625, not the final model.


In [ ]:
pa3('task1', 'train', '--method', 'full')


In [ ]:
pa3('task1', 'evaluate', '--condition', 'full_mid')
pa3('task1', 'qualitative', '--condition', 'full_mid')
pa3('task1', 'evaluate', '--condition', 'full')
pa3('task1', 'qualitative', '--condition', 'full')


### Recover an interrupted run

Use the latest **existing** recovery file for that method. Resume restores optimizer, scheduler, scaler, RNG, and example position, and preserves discarded post-checkpoint logs in backup files. If interrupted before update 312, archive that incomplete method's log directory before starting it again. Do not run the following example unless recovering.


In [ ]:
# pa3('task1', 'train', '--method', 'full', '--resume',
#     str(ARTIFACTS / 'checkpoints/task1/full/update_625.pt'))


## 7. Remaining methods (independent starting checkpoints)

L2-SP uses the unaveraged starting-point penalty with lambda 0.001. LoRA-4/16 use alpha 8/32. OPLoRA-16 uses protection rank 128 and the same initial unprojected factors as LoRA-16. Its exact FP32 SVD preprocessing can take time and is cached.


In [ ]:
pa3('task1', 'train', '--method', 'l2sp')
pa3('task1', 'evaluate', '--condition', 'l2sp')
pa3('task1', 'qualitative', '--condition', 'l2sp')


In [ ]:
pa3('task1', 'train', '--method', 'lora4')
pa3('task1', 'evaluate', '--condition', 'lora4')
pa3('task1', 'qualitative', '--condition', 'lora4')


In [ ]:
pa3('task1', 'train', '--method', 'lora16')
pa3('task1', 'evaluate', '--condition', 'lora16')
pa3('task1', 'qualitative', '--condition', 'lora16')


In [ ]:
pa3('task1', 'train', '--method', 'oplora16')
pa3('task1', 'evaluate', '--condition', 'oplora16')
pa3('task1', 'qualitative', '--condition', 'oplora16')


## 8. Diagnostics and figures

Weight/activation diagnostics use FP32 at down_proj layers 0, 13, 27 and the same cached original-model activations. Summaries and plots are generated from saved measurements only. The final qualitative condition automatically assembles all 168 responses and the 42-row annotation CSV; fill its annotations yourself. Existing human annotations are not overwritten.


In [ ]:
pa3('task1', 'diagnostics')
pa3('task1', 'summarize')
pa3('task1', 'audit')


## Optional: run/resume the entire pipeline

Use this **instead of** manually running every earlier experiment cell. It reuses completed score files/qualitative conditions and resumes methods from their latest recovery checkpoint. It refuses to overwrite a training log when no recovery checkpoint exists. It still requires installation, Drive mounting, and environment variables from sections 1–3.


In [ ]:
# os.environ['PA3_EVAL_BATCH_SIZE'] = '2'
# subprocess.run(['bash', 'scripts/run_task1.sh'], cwd=PROJECT, check=True)


## 9. Inspect evidence

The audit distinguishes missing artifacts and incomplete human annotations. Read `README.md` for the full protocol and output schema. Commit small result/configuration files after the real experiment, not raw datasets or large checkpoints. Write your own report and interpretations, as required by the manual.


In [ ]:
print((ARTIFACTS / 'evidence/task1/status.json').read_text())
